In [1]:
import os
from dotenv import load_dotenv

In [2]:
import pandas as pd
from openai import OpenAI
import json
from datasets import load_dataset

prompt_template = lambda job_description : f"""Read the following job description and create a concise job search query with at most 3 specialized skills or \
areas of expertise that are distinct to the role. Exclude generic data science or software engineering skills like AI, machine \
learning, and coding languages unless they are explicitly highlighted as unique or advanced. Keep the query short and human-like, \
suitable for typing into a search engine. 

Here's the job description: {job_description}"""

/home/kausthubk/win_src/python_playground/text_embedding_fine_tuning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def generate_query(job_description, client: OpenAI):
    prompt = prompt_template(job_description)

    # make api call
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ], 
        temperature = 0.7
    )
    
    # return response
    return response.choices[0].message.content

In [4]:
load_dotenv(dotenv_path='.env')
client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

In [5]:
# load data from HF hub
ds = load_dataset("datastax/linkedin_job_listings")

Generating train split: 100%|██████████| 123849/123849 [00:04<00:00, 25801.16 examples/s]


In [6]:
type(ds)

datasets.dataset_dict.DatasetDict

In [8]:
train_df = ds['train'].to_pandas()
len(train_df)

123849

In [9]:
train_df.columns

Index(['job_id', 'company_name', 'title', 'description', 'max_salary',
       'pay_period', 'location', 'company_id', 'views', 'med_salary',
       'min_salary', 'formatted_work_type', 'applies', 'original_listed_time',
       'remote_allowed', 'job_posting_url', 'application_url',
       'application_type', 'expiry', 'closed_time',
       'formatted_experience_level', 'skills_desc', 'listed_time',
       'posting_domain', 'sponsored', 'work_type', 'currency',
       'compensation_type', 'normalized_salary', 'zip_code', 'fips'],
      dtype='object')

In [12]:
train_df[['title', 'description']].to_parquet('./.data_cache/job_data.parquet')

In [15]:
test_desc = train_df.iloc[0].description
test_desc

'Job descriptionA leading real estate firm in New Jersey is seeking an administrative Marketing Coordinator with some experience in graphic design. You will be working closely with our fun, kind, ambitious members of the sales team and our dynamic executive team on a daily basis. This is an opportunity to be part of a fast-growing, highly respected real estate brokerage with a reputation for exceptional marketing and extraordinary culture of cooperation and inclusion.Who you are:You must be a well-organized, creative, proactive, positive, and most importantly, kind-hearted person. Please, be responsible, respectful, and cool-under-pressure. Please, be proficient in Adobe Creative Cloud (Indesign, Illustrator, Photoshop) and Microsoft Office Suite. Above all, have fantastic taste and be a good-hearted, fun-loving person who loves working with people and is eager to learn.Role:Our office is a fast-paced environment. You’ll work directly with a Marketing team and communicate daily with ot

In [16]:
q = generate_query(job_description=test_desc, client=client)

In [17]:
q

'"Marketing coordination, Adobe Creative Cloud proficiency, event planning"'

In [18]:
# List of strings to search for
search_terms = ["Data Scientist", "Data Analyst", "Machine Learning Engineer", 
                "Data Engineer", "AI Engineer", "Deep Learning"]

# Create a regex pattern to match any of the strings
pattern = '|'.join(search_terms)

# Filter rows that contain any of the search terms
filtered_train_df = train_df[train_df['title'].str.contains(pattern, case=False, na=False)]
train_df.shape, filtered_train_df.shape

((123849, 31), (1179, 31))

In [19]:
filtered_train_df[['title', 'description']].to_parquet('./.data_cache/job_data_ai.parquet')

In [22]:
job_description_list = filtered_train_df['description'].to_list()
len(job_description_list), job_description_list[0]

(1179,
 "Data Engineer with Kafka (W2 Only)💯% Remote\nMin 10 to12+ strong development experience neededVery strong experience in Kafka and Kafka data injection Strong exp in working with API.Strong exp in Python with AWS.Experience with Informatica IICS and Snowflake. Expertise in Snowflake's cloud data platform, including data loading, transformation, and querying using Snowflake SQL.Experience with SQL-based development, optimization, and tuning for large-scale data processing.Strong understanding of dimensional modeling concepts and experience in designing and implementing data models for analytics and reporting purposes.hands-on experience in IICS or Informatica Power Center ETL development1+ years of hands-on experience in Linux and shell scripting.1+ years of experience working with git.1+ years of related industry experience in an enterprise environment.1+ years of hands-on experience in Python programming.\n")

In [23]:
sample = job_description_list[:100]
len(sample)

100

In [24]:
# create batch requests
batch_requests = [
    {
        "custom_id": f"request-{i+1}",  # Custom ID for tracking
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-4o-mini",
            "messages": [
                {"role": "user", "content": prompt_template(jd)}
            ],
            "temperature": 0.7
        }
    }
    for i, jd in enumerate(sample)
]

In [25]:
# Convert to JSONL format (newline-delimited JSON)
batch_jsonl = "\n".join(json.dumps(request) for request in batch_requests)

In [28]:
# Save to a .jsonl file
with open(f"./.data_cache/batch_requests_{len(sample)}.jsonl", "w") as file:
    file.write(batch_jsonl)

In [30]:
batch_input_file = client.files.create(
    file=open("./.data_cache/batch_requests_100.jsonl", "rb"),
    purpose="batch"
)

print(batch_input_file)

FileObject(id='file-1NkywkUTKosTKLGvH5Y8mE', bytes=373922, created_at=1764125057, filename='batch_requests_100.jsonl', object='file', purpose='batch', status='processed', expires_at=1766717057, status_details=None)


In [ ]:
# # create batch job
# batch_object = client.batches.create(
#     input_file_id=batch_input_file.id,
#     endpoint="/v1/chat/completions",
#     completion_window="24h",
#     metadata={
#         "description": "synthetic queries from job descriptions"
#     }
# )